# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
import json
warnings.filterwarnings('ignore')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their fields, and their `@id`s.

In [ ]:
# For this dataset, reading all record sets and field @ids
record_sets = list(dataset.record_sets)

for record_set in record_sets:
    print(f"RecordSet @id: {record_set['@id']}, name: {record_set.get('name','(no name)')}")
    print("  Fields:")
    for field in record_set['fields']:
        print(f"    Field @id: {field['@id']} | name: {field.get('name','(no name)')} | dataType: {field.get('dataType','(no dataType)')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from each record set
# List record set @ids found in the overview
record_set_ids = [r['@id'] for r in record_sets]
print(f"Available record set @ids: {record_set_ids}")

# For demonstration, select the primary protein quantification record set (usually holding main table of interest)
selected_record_set_id = record_set_ids[0]  # You may change this after reviewing the record sets above

dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records from {record_set_id}")
    except Exception as e:
        print(f"Could not load records from record set {record_set_id}: {e}")

if selected_record_set_id in dataframes:
    print(f"Fields (column names) in '{selected_record_set_id}':")
    print(list(dataframes[selected_record_set_id].columns))
    display(dataframes[selected_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section can include removing outliers, transforming data distributions, or grouping data by key attributes for further analysis.

In [ ]:
import numpy as np

# Use the DataFrame for the main table
df = dataframes[selected_record_set_id]

# Select a numeric field for analysis. Replace with your actual numeric field's @id or column name if needed.
# For demonstration, we look for a likely numeric field (e.g., coverage, peptide count, molecular weight, abundance, etc.)
possible_numeric_fields = [c for c in df.columns if df[c].dtype in [np.float64, np.float32, np.int64, np.int32]]
if not possible_numeric_fields:
    # Try to cast fields to float, for those failing, skip
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
            if df[col].dtype in [np.float64, np.float32, np.int64, np.int32]:
                possible_numeric_fields.append(col)
        except Exception:
            pass

if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]  # Choose the first numeric field found
else:
    raise RuntimeError('No numeric fields found in the main DataFrame.')

print(f"Using numeric field '{numeric_field}' for filtering and normalization.")

threshold = df[numeric_field].mean() if df[numeric_field].mean() > 0 else 10

filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / (filtered_df[numeric_field].std() if filtered_df[numeric_field].std() else 1)
print(f"\nNormalized '{numeric_field}' for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by likely a categorical field (e.g., 'sample', 'type', etc.; pick first non-numeric, if exists)
possible_group_fields = [c for c in df.columns if df[c].dtype == object and c != numeric_field]
group_field = possible_group_fields[0] if possible_group_fields else None
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"\nGrouped data by '{group_field}' (mean of '{numeric_field}'):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the numeric field
plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field].dropna(), kde=True)
plt.title(f"Distribution of '{numeric_field}'")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If group_field exists, visualize mean by group
if group_field:
    plt.figure(figsize=(10,5))
    order = filtered_df[group_field].value_counts().index
    sns.barplot(data=filtered_df, x=group_field, y=numeric_field, order=order)
    plt.title(f"Mean {numeric_field} by {group_field} (filtered)")
    plt.xlabel(group_field)
    plt.ylabel(f"Mean {numeric_field}")
    plt.xticks(rotation=45, ha='right')
    plt.show()

## 6. Conclusion
In this notebook, we've loaded the FAIR² dataset using its Croissant schema, explored the available record sets and fields (referenced by their `@id`), extracted records into pandas DataFrames, performed basic EDA such as filtering and normalization on a selected numeric field, grouped by a categorical attribute, and visualized data distributions.

To explore further, inspect additional `record_set @id`s and field @ids, analyze relationships between variables, and apply domain-specific processing as appropriate for mass spectrometry protein data. All references use the dataset's unique identifiers for transparency and reproducibility.